<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/Psep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import time

# ══════════════════════════════════════════════════════════════════════════════
# PARAMÈTRES
# Cible : deux hyperplans parallèles {P_sep = 0} et {P_sep = ε} (P_sep linéaire),
# label sigmoïde séparant les deux plans. Méthode : experts cylindriques
# (gaussiennes anisotropes) + Wendland anisotropes, UN σ aléatoire par dimension
# et par centre, SANS décroissance géométrique de σ.
# ══════════════════════════════════════════════════════════════════════════════
params = {
    # ── GÉOMÉTRIE ─────────────────────────────────────────────────────────────
    "n_ambiant":    4,        # dimension ambiante d
    "deg_P":        4,        # degré de P_sep (1 = hyperplans ; >1 = nappes courbes)
    "n_terms_poly": 20,       # nb de monômes non nuls de P_sep (sparsité)
    "domain":       (0.0, 10),   # cube [lo,hi]^d où sont tirés les points
    "epsilon":      1,      # écart des deux niveaux : {P_sep=0} et {P_sep=ε}
    "proj_iter":    50,       # itérations max de Gauss-Newton pour projeter
    "proj_tol":     1e-10,    # tolérance |P(x)−t| pour accepter un point

    # ── DONNÉES ───────────────────────────────────────────────────────────────
    "n_train":      10,
    "n_test":       5000,
    "seeds": {42: 48124, 43: 24322, 44: 12561, 99: 9269},

    # ── NORME DE SOBOLEV ──────────────────────────────────────────────────────
    # w0 : petit poids L² — indispensable numériquement (ancre la constante et
    #      relève le noyau de G ; sans lui le QP dérive, cf. offset qui explose).
    "weights":  {0: 1e-6, 1: 1, 2: 0.5, 3: 0.0},

    # ── EXPERTS ───────────────────────────────────────────────────────────────
    "n_levels":        0,     # experts cylindriques (gaussiennes anisotropes)
    "n_levels_wnd":    0,     # experts Wendland anisotropes (support compact)
    "deg_poly_expert": 15,     # expert polynomial ; -1 pour le désactiver
    "n_dict":          5000,  # dictionnaire tiré par expert noyau
    "n_centres":       800,   # centres retenus par expert noyau
    "sigma_min":       0.2,
    "sigma_max":       1.0,

    # ── SEMI-INTERPOLATION (QP) ───────────────────────────────────────────────
    #   min ‖u‖²_H   s.c.   y_i·u(x_i) >= qp_margin
    # Convexe, solution unique et stable. Pas de régularisation : la norme H est
    # l'objectif, les contraintes remplacent le terme de données.
    "qp_margin": 0.1,         # marge dure
    "a_label":   0.1,         # labels ±a_label (cohérent avec qp_margin)
    "thres1":    1e-19,       # bande spectrale de la base H-orthonormée :
    "thres2":    1,         #   on garde les modes thres1 < s/s_max < thres2
    "const_pen": 0,        # pénalité sur la constante libre. 0 = totalement
                              #   libre -> l'offset dérive (AUC bonne, accuracy 0.5).
                              #   Un petit poids l'ancre sans l'empêcher de caler.

    # ── GRAM ──────────────────────────────────────────────────────────────────
    "n_G":    2000,           # points cloud pour la Gram de chaque expert
    "n_G_H":  5000,           # points cloud pour la Gram finale sur H
}
params["n_unlabeled"]        = 4000 - params["n_train"]
params["train_center_ratio"] = 0    # tous les centres tirés côté train+unlabeled

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d)
                   if sum(exp) <= degree]
    # borne : on ne peut pas tirer plus de monômes qu'il n'en existe au degré donné
    # (ex. degré 1 en dim 4 -> seulement 5 monômes, dont la constante).
    n_terms = min(n_terms, len(all_indices))
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:,j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices)
                   if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:,j]**e
                y += c * term
            return y
        return Pn
    return P

d   = params["n_ambiant"]
eps = params["epsilon"]

# P_sep POLYNOMIAL de degré params["deg_P"] (normalisé).
# Les deux nappes {P_sep=0} et {P_sep=ε} sont des HYPERSURFACES de niveau.
# Pour deg_P=1 ce sont des hyperplans parallèles ; au-delà, des variétés courbes.
P_sep, idx_sep, coeffs_sep = random_sparse_polynomial(
    d, params["deg_P"], params["n_terms_poly"], seed=params["seeds"][99])
P_sep = normalize_polynomial(P_sep, idx_sep, coeffs_sep)

# ── RENORMALISATION SUR LE DOMAINE ────────────────────────────────────────────
# normalize_polynomial normalise les COEFFICIENTS, pas la plage de valeurs sur le
# cube. À deg_P=3 sur [0,10]^4, les termes cubiques font exploser P (std ~330),
# et ε=0.7 devient invisible : les deux nappes se collent (0.2% du cube entre
# elles) -> classes superposées, tout le monde à 0.50.
# On rescale P pour que std(P) = 1 sur le domaine : ε=0.7 garde alors le même
# sens géométrique quel que soit deg_P.
_Xs = np.random.default_rng(12345).uniform(*params["domain"], (200000, d))
_P_raw = P_sep
_scale = float(_P_raw(_Xs).std())
_shift = float(_P_raw(_Xs).mean())
P_sep = lambda X, _f=_P_raw, _s=_scale, _m=_shift: (_f(X) - _m) / _s
print(f"  P_sep renormalisé sur le domaine : std=1 (facteur {_scale:.3g}, centre {_shift:.3g})")

# ══════════════════════════════════════════════════════════════════════════════
# CLOUD sur {Q=0} = {P_sep=0} ∪ {P_sep=ε}   —  PROJECTION GAUSS-NEWTON
#
# P_sep n'est plus affine : pas de projection analytique. On projette un point x
# sur la nappe {P_sep = t} par itérations de Gauss-Newton le long du gradient :
#       x <- x − ((P(x) − t)/‖∇P(x)‖²) · ∇P(x)
# C'est la direction de plus forte variation : chaque pas annule P(x)−t au 1er
# ordre. Converge quadratiquement près de la nappe. (Pour deg_P=1, ∇P est constant
# et UN pas suffit : on retrouve exactement la projection analytique d'avant.)
#
# On NE projette PAS sur Q = P·(P−ε) : ∇Q = (2P−ε)∇P s'annule sur {P=ε/2}, ce qui
# piégeait les points sur la nappe médiane (bug historique). On projette sur
# chaque nappe séparément, en tirant 50/50 -> équilibre exact garanti.
# ══════════════════════════════════════════════════════════════════════════════
DOM_LO, DOM_HI = params["domain"]

def grad_P(X, h=1e-6):
    """Gradient de P_sep par différences finies centrées (P est une boîte noire)."""
    n, dd = X.shape
    G = np.empty((n, dd))
    for k in range(dd):
        Xp = X.copy(); Xp[:, k] += h
        Xm = X.copy(); Xm[:, k] -= h
        G[:, k] = (P_sep(Xp) - P_sep(Xm)) / (2*h)
    return G

def project_to_level(X, t, n_iter=params["proj_iter"], tol=params["proj_tol"]):
    """Projection Gauss-Newton de X sur la nappe {P_sep = t}."""
    Xp = X.copy()
    for _ in range(n_iter):
        r = P_sep(Xp) - t                       # résidu
        if np.max(np.abs(r)) < tol:
            break
        G = grad_P(Xp)
        g2 = np.sum(G**2, axis=1)
        g2 = np.where(g2 > 1e-14, g2, 1e-14)    # garde-fou : gradient ~nul
        Xp = Xp - (r/g2)[:, None] * G
    return Xp

def sample_on_level(t, n_target, seed, max_tries=2000000):
    """Points sur {P_sep=t} ∩ [lo,hi]^d, par tirage + projection Gauss-Newton.
    On ne garde que les points restés dans le cube ET effectivement convergés."""
    rng = np.random.default_rng(seed); out = []; got = 0; tries = 0
    while got < n_target and tries < max_tries:
        m = max(8 * (n_target - got), 1000)
        X  = rng.uniform(DOM_LO, DOM_HI, (m, d))
        Xp = project_to_level(X, t)
        inbox = np.all((Xp >= DOM_LO) & (Xp <= DOM_HI), axis=1)
        conv  = np.abs(P_sep(Xp) - t) < params["proj_tol"]   # rejet des non-convergés
        ok = inbox & conv
        g  = Xp[ok]
        if len(g): out.append(g); got += len(g)
        tries += m
    if got < n_target:
        raise RuntimeError(f"nappe t={t}: seulement {got}/{n_target} points collectés "
                           f"(augmenter proj_iter, ou domaine/deg_P inadaptés)")
    return np.vstack(out)[:n_target]

def sample_two_levels(n_target, seed):
    """n_target points répartis 50/50 EXACTEMENT sur les deux nappes, mélangés."""
    n0 = n_target // 2; n1 = n_target - n0
    X0 = sample_on_level(0.0, n0, seed)
    X1 = sample_on_level(eps, n1, seed + 1)
    X  = np.vstack([X0, X1])
    rng = np.random.default_rng(seed + 2)
    return X[rng.permutation(len(X))]

print(f"Génération du cloud sur {{Q=0}} = {{P_sep=0}} ∪ {{P_sep=ε}} "
      f"(deg_P={params['deg_P']}, projection Gauss-Newton, 50/50)...")
X_train     = sample_two_levels(params["n_train"],     seed=params["seeds"][42])
X_test      = sample_two_levels(params["n_test"],      seed=params["seeds"][43])
X_unlabeled = sample_two_levels(params["n_unlabeled"], seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_test:{X_test.shape}  X_unlabeled:{X_unlabeled.shape}")

for nom, Xc in [("train", X_train), ("test", X_test)]:
    p = P_sep(Xc)
    n0 = int((np.abs(p) < eps/2).sum()); n1 = int((np.abs(p-eps) < eps/2).sum())
    resid = np.minimum(np.abs(p), np.abs(p-eps)).max()
    print(f"  {nom:5s} : nappe0={n0}  nappeε={n1}  max|résidu projection|={resid:.2e}")

# ── Cible : label ±a_label selon le plan (plateau haut / plateau bas) ──────────
# CLASSIFICATION : y = +a_label sur {P_sep=ε}, −a_label sur {P_sep=0}. La loss
# log-cosh sature loin de la frontière : seul le côté du seuil (0) compte.
A_LABEL = params["a_label"]       # labels ±a_label
def target_label(X):
    return np.where(P_sep(X) > eps/2, A_LABEL, -A_LABEL)

y_train = target_label(X_train); y_test = target_label(X_test)
# Labels ±1 pour les BASELINES (RBF, Ridge poly) : leur échelle naturelle.
# Imposer ±a_label aux baselines biaiserait la comparaison (leur régularisation
# interne est calibrée pour des cibles O(1)). La décision étant sign(f), l'échelle
# du label est un paramètre interne à chaque méthode -> chacun sur la sienne.
y_train_pm1 = np.sign(y_train); y_test_pm1 = np.sign(y_test)

# ── Échelle de travail de la MÉTHODE H ────────────────────────────────────────
# sigmoid : f est ajusté sur des labels ±1 (β règle la marge sur l'échelle de f),
#           donc TOUT le pipeline H (fit, sélection, affichage, MSE) est en ±1.
# hinge2/logcosh/mse : la marge/cible est a_label, pipeline en ±a_label.
# QP : labels ±1, cohérents avec la marge y_i·u(x_i) >= qp_margin
yH_train, yH_test = y_train_pm1, y_test_pm1
n_pos = int((y_train > 0).sum()); n_neg = int((y_train < 0).sum())
print(f"  y_train: {n_neg} × (−{A_LABEL:g})  |  {n_pos} × (+{A_LABEL:g})   "
      f"(labels ±a_label, décision sign(f))")

X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)

# ══════════════════════════════════════════════════════════════════════════════
# ÉCHANTILLONNAGE DES CENTRES ET SIGMAS (anisotrope, UN σ aléatoire par dim/centre,
# log-uniforme dans [sigma_min, sigma_max], SANS décroissance)
# ══════════════════════════════════════════════════════════════════════════════
def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates - n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    centers = np.vstack(parts)
    log_s = rng.uniform(np.log(params["sigma_min"]), np.log(params["sigma_max"]),
                        size=(len(centers), d))
    sigmas = np.exp(log_s)
    return centers, sigmas

# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES CYLINDRIQUES (anisotropes) : features + dérivées analytiques
# ══════════════════════════════════════════════════════════════════════════════
def cyl_features(X, centers, sigmas):
    diff = X[:,None,:] - centers[None,:,:]
    sq_w = np.sum(diff**2 / sigmas[None,:,:]**2, axis=2)
    return np.exp(-sq_w)

def cyl_derivatives(X, centers, sigmas, weights=None):
    d_ = X.shape[1]
    diff = X[:,None,:] - centers[None,:,:]
    inv_s2 = 1./sigmas**2
    diff_w = diff * inv_s2[None,:,:]
    sq_w = np.sum(diff*diff_w, axis=2)
    phi = np.exp(-sq_w)
    grad = -2*diff_w*phi[:,:,None]
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id = np.eye(d_)
        diag  = -2*inv_s2[None,:,:,None]*Id[None,None,:,:]
        cross = 4*np.einsum('nmi,nmj->nmij', diff_w, diff_w)
        hess  = (diag+cross)*phi[:,:,None,None]
    else: hess = None
    if need_third:
        Id = np.eye(d_)
        inv_s2_bc = inv_s2[None,:,:]
        cubic = -8*np.einsum('nmi,nmj,nmk->nmijk', diff_w, diff_w, diff_w)
        sym   = 4*(np.einsum('ij,nmi,nmk->nmijk', Id, inv_s2_bc, diff_w)+
                   np.einsum('ik,nmi,nmj->nmijk', Id, inv_s2_bc, diff_w)+
                   np.einsum('jk,nmj,nmi->nmijk', Id, inv_s2_bc, diff_w))
        third = (cubic+sym)*phi[:,:,None,None,None]
    else: third = None
    return phi, grad, hess, third

# ══════════════════════════════════════════════════════════════════════════════
# WENDLAND ANISOTROPES : support compact {r<1}, r²=Σ (x_k−c_k)²/σ_k².
# Profil C^{2k} minimal tel que 2k ≥ ordre max des poids (ordre 3 -> C⁴).
# ══════════════════════════════════════════════════════════════════════════════
def _wnd_profile(max_order):
    k = max(1, int(np.ceil(max_order/2)))
    sd = lambda num, r: np.where(r > 1e-12, num/np.maximum(r, 1e-12), 0.0)
    if k == 1:      # C²
        return (lambda r,rp: rp**4*(4*r+1),
                lambda r,rp: -20*rp**3,
                lambda r,rp: sd(60*rp**2, r),
                None)
    elif k == 2:    # C⁴
        return (lambda r,rp: rp**6*(35*r**2+18*r+3)/3,
                lambda r,rp: -(56/3)*rp**5*(5*r+1),
                lambda r,rp: 560.*rp**4,
                lambda r,rp: sd(-2240.*rp**3, r))
    else:           # C⁶
        return (lambda r,rp: rp**8*(32*r**3+25*r**2+8*r+1),
                lambda r,rp: -22*rp**7*(16*r**2+7*r+1),
                lambda r,rp: 528*rp**6*(6*r+1),
                lambda r,rp: -22176.*rp**5)

_wnd_max_order = max([o for o, w in params["weights"].items() if w != 0])
WND_PHI, WND_H1, WND_H2, WND_H3 = _wnd_profile(_wnd_max_order)

def wnd_features(X, centers, sigmas):
    u = (X[:,None,:]-centers[None,:,:])/sigmas[None,:,:]
    r = np.sqrt(np.sum(u**2, axis=2)); rp = np.maximum(1.-r, 0.)
    return WND_PHI(r, rp)

def wnd_derivatives(X, centers, sigmas, weights=None):
    d_ = X.shape[1]
    u  = (X[:,None,:]-centers[None,:,:])/sigmas[None,:,:]
    r  = np.sqrt(np.sum(u**2, axis=2)); rp = np.maximum(1.-r, 0.)
    inv_s  = 1./sigmas
    inv_s2 = inv_s**2
    us  = u*inv_s[None,:,:]
    phi = WND_PHI(r, rp)
    H1  = WND_H1(r, rp)
    grad = H1[:,:,None]*us
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id = np.eye(d_)
        H2 = WND_H2(r, rp)
        hess = (H2[:,:,None,None]*np.einsum('nmi,nmj->nmij', us, us)
                + H1[:,:,None,None]*(Id[None,None,:,:]*inv_s2[None,:,:,None]))
    else: hess = None
    if need_third:
        assert WND_H3 is not None, "poids d'ordre 3 : profil Wendland C⁴ minimum requis"
        Id = np.eye(d_)
        H3 = WND_H3(r, rp)
        cubic = H3[:,:,None,None,None]*np.einsum('nmi,nmj,nmk->nmijk', us, us, us)
        sym = (np.einsum('ij,mi,nmk->nmijk', Id, inv_s2, us)
              +np.einsum('ik,mi,nmj->nmijk', Id, inv_s2, us)
              +np.einsum('jk,mj,nmi->nmijk', Id, inv_s2, us))
        third = cubic + WND_H2(r, rp)[:,:,None,None,None]*sym
    else: third = None
    return phi, grad, hess, third

def build_G(phi, grad, hess, third, weights, n):
    G = np.zeros((phi.shape[1], phi.shape[1]))
    w0 = weights.get(0,0.)
    if w0: G += w0*(phi.T@phi)/n
    w1 = weights.get(1,0.)
    if w1 and grad  is not None: G += w1*np.einsum('xik,xjk->ij',     grad, grad,  optimize='optimal')/n
    w2 = weights.get(2,0.)
    if w2 and hess  is not None: G += w2*np.einsum('xikl,xjkl->ij',   hess, hess,  optimize='optimal')/n
    w3 = weights.get(3,0.)
    if w3 and third is not None: G += w3*np.einsum('xiklm,xjklm->ij', third,third, optimize='optimal')/n
    return G

def sobolev_norm2(derivs, weights, idx=None):
    """||f||²_H = Σ_o w_o · mean_x ||D^o f(x)||²_F pour une fonction déjà évaluée."""
    tot = 0.
    for key, order, axes in (('phi',0,()), ('grad',1,(1,)),
                             ('hess',2,(1,2)), ('third',3,(1,2,3))):
        w = weights.get(order, 0.)
        if not w:            continue
        T = derivs.get(key)
        if T is None:        continue
        if idx is not None:  T = T[idx]
        tot += w * (np.mean(T**2) if order == 0
                    else np.mean(np.sum(T**2, axis=axes)))
    return tot

# ══════════════════════════════════════════════════════════════════════════════
# SOLVEUR QP : semi-interpolation Sobolev sur le polyèdre de marge
#
#   min_c  cᵀ G c    s.c.   y_i (A c)_i >= a   pour tout i du train
#
# = projection, au sens de la norme H, sur le convexe E = {u : y_i u(x_i) >= a}.
# Forme quadratique CONVEXE sur un polyèdre (n contraintes = n_train, très peu).
# Contrairement aux loss (sigmoïde non convexe, etc.), la solution est UNIQUE et
# STABLE : pas de minimum parasite, pas de fuite. Si le plateau (norme H ~0) est
# dans le span des atomes, le QP le trouve. Si E est vide (dico incapable de
# séparer avec marge a), le QP est INFAISABLE -> diagnostic net.
# ══════════════════════════════════════════════════════════════════════════════
from scipy.optimize import minimize, LinearConstraint

def solve_qp(A, y, G, margin, verbose=False):
    """min cᵀGc s.c. diag(y)Ac >= margin. Retourne (c, faisable, rang_effectif)."""
    n, k = A.shape
    Gr = G + 1e-10 * np.eye(k)            # définie positive
    C  = np.diag(y) @ A                    # contraintes y_i (Ac)_i >= margin
    con = LinearConstraint(C, lb=margin, ub=np.inf)
    # point de départ admissible : moindres carrés vers (marge·y) amplifié
    try:
        c0 = np.linalg.lstsq(A, 1.5*margin*y, rcond=None)[0]
    except Exception:
        c0 = np.zeros(k)
    res = minimize(lambda c: c@Gr@c, c0, jac=lambda c: 2*Gr@c,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-11})
    c = res.x
    marge_eff = float(np.min(y * (A @ c)))
    faisable  = marge_eff >= margin - 1e-4
    if verbose:
        print(f"      QP: succès={res.success} marge_atteinte={marge_eff:.3f} "
              f"||u||_H={np.sqrt(max(c@G@c,0)):.4f}")
    # rang effectif : nb de contraintes actives (à la marge)
    rank_eff = int(np.sum(np.abs(y*(A@c) - margin) < 1e-3))
    return c, faisable, max(rank_eff, 1)

def method_metrics(f_train, f_test, coeffs, G, yH_tr, yH_te):
    """QP (semi-interpolation) : on affiche la marge minimale atteinte (les
    contraintes sont-elles satisfaites) et ‖u‖_H (l'objectif minimisé).
    Accuracy : sign(0)=0 ne matche aucun label ±1 -> une fonction nulle compterait
    0.000 (trompeur, c'est du hasard) ; on la compte 0.5."""
    marge_min = float(np.min(yH_tr * f_train)) if len(f_train) else np.nan
    nH = float(np.sqrt(max(coeffs @ G @ coeffs, 0))) if coeffs is not None else np.nan
    def _acc(f, y):
        sg = np.sign(f)
        return np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float)))
    return marge_min, nH, _acc(f_train, yH_tr), _acc(f_test, yH_te)

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : experts cylindriques + Wendland indépendants
# Chaque f_i = argmin sur son propre dictionnaire aléatoire (σ sans decay)
# ══════════════════════════════════════════════════════════════════════════════
level_specs = ([("cyl", i) for i in range(params["n_levels"])] +
               [("wnd", i) for i in range(params["n_levels_wnd"])])
n_experts_total = len(level_specs)
expert_names    = [f"{typ}_{i+1:02d}" for typ, i in level_specs]

print(f"\nPhase 1 : calcul de {n_experts_total} experts "
      f"({params['n_levels']} cyl + {params['n_levels_wnd']} "
      f"wnd C{2*max(1,int(np.ceil(_wnd_max_order/2)))})...")

F_train = np.zeros((len(X_train), n_experts_total))
F_test  = np.zeros((len(X_test),  n_experts_total))
F_all   = np.zeros((N_all,        n_experts_total))
fi_derivs_list = []
times_p1 = []

for slot, (lev_type, lev_idx) in enumerate(level_specs):
    t0 = time.time()
    rng = np.random.default_rng(seed={"cyl":0, "wnd":2000}[lev_type]+lev_idx)

    feat_fn  = wnd_features    if lev_type=="wnd" else cyl_features
    deriv_fn = wnd_derivatives if lev_type=="wnd" else cyl_derivatives

    candidates, sigmas_cand = sample_candidates(
        X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"], rng)

    A_cand = feat_fn(X_train, candidates, sigmas_cand)

    # ── SÉLECTION DES CENTRES : sous-échantillon aléatoire, INDÉPENDANT de y ──
    # (un filtre par corrélation avec y sur 10 points ne garderait que les atomes
    #  collés aux points d'entraînement -> sur-apprentissage fabriqué en amont.)
    k = min(params["n_centres"], len(candidates))
    idx_sel = rng.choice(len(candidates), size=k, replace=False)
    centers_sel = candidates[idx_sel]
    sigmas_sel  = sigmas_cand[idx_sel]
    A     = A_cand[:, idx_sel]
    Atest = feat_fn(X_test, centers_sel, sigmas_sel)
    Aall  = feat_fn(X_all,  centers_sel, sigmas_sel)

    phi, grad, hess, third = deriv_fn(X_all, centers_sel, sigmas_sel, params["weights"])

    n_G   = min(N_all, params["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G(phi[idx_G],
                grad[idx_G]  if grad  is not None else None,
                hess[idx_G]  if hess  is not None else None,
                third[idx_G] if third is not None else None,
                params["weights"], n_G)

    coeffs, feas, rank_p1 = solve_qp(A, yH_train, G, params["qp_margin"])
    mask = np.ones(rank_p1, bool)
    if not feas:
        print("      [QP] INFAISABLE : ce dictionnaire ne sépare pas avec la marge")

    F_train[:,slot] = A     @ coeffs
    F_test [:,slot] = Atest @ coeffs
    F_all  [:,slot] = Aall  @ coeffs

    fi_derivs_list.append({
        'phi':   phi   @ coeffs,
        'grad':  np.einsum('xjk,j->xk',    grad, coeffs) if grad  is not None else None,
        'hess':  np.einsum('xjkl,j->xkl',  hess, coeffs) if hess  is not None else None,
        'third': np.einsum('xjklm,j->xklm',third,coeffs) if third is not None else None,
    })

    data_i, reg_i, acc_tr_i, acc_te_i = method_metrics(
        F_train[:,slot], F_test[:,slot], coeffs, G, yH_train, yH_test)
    loss_total = data_i + reg_i
    t1 = time.time(); times_p1.append(t1-t0)
    print(f"  {expert_names[slot]:7s} | rang={mask.sum():3d}/{k} | "
          f"loss={loss_total:.6f} (data={data_i:.6f} reg={reg_i:.6f}) | "
          f"acc_tr={acc_tr_i:.2f} acc_te={acc_te_i:.3f} | "
          f"σ∈[{sigmas_sel.min():.2f},{sigmas_sel.max():.2f}] | {times_p1[-1]:.1f}s")

# ══════════════════════════════════════════════════════════════════════════════
# EXPERT POLYNOMIAL (optionnel : deg_poly_expert >= 0)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import PolynomialFeatures

USE_POLY_EXPERT = params["deg_poly_expert"] >= 0
print("\nCalibration expert polynomial..." if USE_POLY_EXPERT
      else "\nExpert polynomial DÉSACTIVÉ (deg_poly_expert < 0)")

rng_H = np.random.default_rng(seed=999)
idx_H = rng_H.choice(N_all, size=min(N_all, params["n_G_H"]), replace=False)
n_H   = len(idx_H)
X_H   = X_all[idx_H]

w1 = params["weights"].get(1, 0.)
w2 = params["weights"].get(2, 0.)
w3 = params["weights"].get(3, 0.)

if USE_POLY_EXPERT:
    deg_e   = params["deg_poly_expert"]
    d_      = params["n_ambiant"]

    # ── Normalisation du domaine [lo,hi] -> [-1,1] pour le CONDITIONNEMENT ──────
    # À degré 13 sur [0,50], x^13 ~ 1e22 : la base monomiale est numériquement
    # morte. On travaille en u = (x-lo)/(hi-lo)*2 - 1 ∈ [-1,1]. Les dérivées ∂/∂x
    # portent alors un facteur SCALE = 2/(hi-lo) par ordre, appliqué pour que la
    # norme Sobolev du polynôme soit dans les MÊMES coordonnées x que les noyaux.
    SCALE = 2.0 / (DOM_HI - DOM_LO)
    def to_u(X): return (X - DOM_LO) * SCALE - 1.0

    poly_e  = PolynomialFeatures(degree=deg_e, include_bias=False)
    Phi_tr  = poly_e.fit_transform(to_u(X_train))     # features en coord. normalisées
    powers_e = poly_e.powers_
    n_feat  = Phi_tr.shape[1]

    def poly_deriv_eval_u(U, powers, coefs, dims=()):
        """Dérivée ∂/∂u_dims du polynôme(coefs) évalué en U (coord. normalisées)."""
        P = powers.astype(np.float64).copy()
        mult = coefs.astype(np.float64).copy()
        for m in dims:
            mult = mult * P[:, m]
            P[:, m] -= 1
        valid = mult != 0
        if not np.any(valid):
            return np.zeros(len(U))
        Up = U[:, None, :] ** P[None, valid, :]
        return Up.prod(axis=2) @ mult[valid]

    # ── Construire la Gram Sobolev G_poly du dictionnaire polynomial ───────────
    # sur les n_G points de Gram (comme les noyaux). Base = colonnes de Phi.
    # Pour chaque feature j (coef = e_j), on évalue ses dérivées ∂/∂x = SCALE^o ∂/∂u.
    rngP   = np.random.default_rng(seed=4242)
    n_Gp   = min(N_all, params["n_G"])
    idx_Gp = rngP.choice(N_all, size=n_Gp, replace=False)
    U_G    = to_u(X_all[idx_Gp])

    # tenseurs (n_Gp, n_feat, [d]^o) des dérivées de CHAQUE feature, en coord. x
    E = np.eye(n_feat)
    PHI_G  = poly_e.transform(U_G)                              # (n_Gp, n_feat)
    GRAD_G = None; HESS_G = None; THIRD_G = None
    if w1:
        GG = np.zeros((n_Gp, n_feat, d_))
        for j in range(n_feat):
            for a in range(d_):
                GG[:, j, a] = SCALE * poly_deriv_eval_u(U_G, powers_e, E[j], (a,))
        GRAD_G = GG
    if w2:
        HG = np.zeros((n_Gp, n_feat, d_, d_))
        for j in range(n_feat):
            for a in range(d_):
                for b in range(a, d_):
                    v = SCALE**2 * poly_deriv_eval_u(U_G, powers_e, E[j], (a,b))
                    HG[:,j,a,b] = v; HG[:,j,b,a] = v
        HESS_G = HG
    if w3:
        from itertools import permutations as _perm
        TG = np.zeros((n_Gp, n_feat, d_, d_, d_))
        for j in range(n_feat):
            for a in range(d_):
                for b in range(a, d_):
                    for cc in range(b, d_):
                        v = SCALE**3 * poly_deriv_eval_u(U_G, powers_e, E[j], (a,b,cc))
                        for p in set(_perm((a,b,cc))):
                            TG[:,j,p[0],p[1],p[2]] = v
        THIRD_G = TG

    G_poly = build_G(PHI_G, GRAD_G, HESS_G, THIRD_G, params["weights"], n_Gp)  # (n_feat,n_feat)

    # ── Semi-interpolation QP dans la BASE H-ORTHONORMÉE ──────────────────────
    # T = V/√s : dans cette base ‖u‖²_H = ‖α‖² (Gram = Id) -> QP bien conditionné,
    # coefficients O(1) au lieu de O(1e9) en monômes bruts.
    # Centrage des features sur le cloud + constante LIBRE (norme H nulle) : elle
    # cale le seuil de décision. w0>0 l'empêche de dériver (sinon offset explose).
    #   min ‖α‖²  s.c.  diag(y)[(Φ̃_tr T) α + β·1] >= qp_margin
    mean_phi = PHI_G.mean(axis=0)
    Phi_tr_c = Phi_tr - mean_phi
    s_g, V_g = np.linalg.eigh(G_poly)
    smax = s_g.max()
    keep = (s_g > smax*params["thres1"]) & (s_g < smax*params["thres2"])
    if keep.sum() == 0:
        raise ValueError(f"bande spectrale vide : ajuster thres1/thres2 "
                         f"(s ∈ [{s_g.min():.2e}, {s_g.max():.2e}])")
    Tmat = V_g[:, keep] / np.sqrt(s_g[keep])
    Phi_tr_o = Phi_tr_c @ Tmat
    r_o = int(keep.sum())
    A_qp = np.hstack([Phi_tr_o, np.ones((len(Phi_tr_o), 1))])
    G_qp = np.eye(r_o + 1); G_qp[-1, -1] = params["const_pen"]
    sol, feas_p, rank_p = solve_qp(A_qp, yH_train, G_qp, params["qp_margin"], verbose=True)
    alpha_o = sol[:r_o]; beta_const = sol[-1]
    coef_poly = Tmat @ alpha_o
    const_offset = beta_const - mean_phi @ coef_poly
    print(f"      [QP poly] base H-ortho rang {r_o}/{n_feat}, "
          f"max|α|={np.abs(alpha_o).max():.2e}, offset={const_offset:.3f}, faisable={feas_p}")

    pred_poly_train = Phi_tr @ coef_poly
    pred_poly_test  = poly_e.transform(to_u(X_test)) @ coef_poly

    # ── Dérivées du polynôme AJUSTÉ sur X_H (coord. x) pour la Gram de Phase 2 ──
    U_H = to_u(X_H)
    delta_phi_poly = poly_e.transform(U_H) @ coef_poly
    if w1 or w2 or w3:
        delta_grad_poly = SCALE * np.stack(
            [poly_deriv_eval_u(U_H, powers_e, coef_poly, (a,)) for a in range(d_)], axis=1)
    else:
        delta_grad_poly = None
    if w2 or w3:
        delta_hess_poly = np.zeros((n_H, d_, d_))
        for a in range(d_):
            for b in range(a, d_):
                v = SCALE**2 * poly_deriv_eval_u(U_H, powers_e, coef_poly, (a,b))
                delta_hess_poly[:,a,b] = v; delta_hess_poly[:,b,a] = v
    else:
        delta_hess_poly = None
    if w3:
        from itertools import permutations
        delta_third_poly = np.zeros((n_H, d_, d_, d_))
        for a in range(d_):
            for b in range(a, d_):
                for cc in range(b, d_):
                    v = SCALE**3 * poly_deriv_eval_u(U_H, powers_e, coef_poly, (a,b,cc))
                    for p in set(permutations((a,b,cc))):
                        delta_third_poly[:,p[0],p[1],p[2]] = v
    else:
        delta_third_poly = None

    data_p, reg_p, acc_tr_p, acc_te_p = method_metrics(
        pred_poly_train, pred_poly_test, coef_poly, G_poly, yH_train, yH_test)
    print(f"  Poly{deg_e}  | feat={n_feat:4d} rang={rank_p:4d} | "
          f"loss={data_p+reg_p:.6f} (data={data_p:.6f} reg={reg_p:.6f}) | "
          f"acc_tr={acc_tr_p:.2f} acc_te={acc_te_p:.3f}")

    F_train = np.column_stack([F_train, pred_poly_train])
    F_test  = np.column_stack([F_test,  pred_poly_test])
    fi_derivs_list.append({
        'phi': delta_phi_poly,   'grad': delta_grad_poly,
        'hess': delta_hess_poly, 'third': delta_third_poly})
    expert_names += [f"Poly{deg_e}"]

print(f"  H étendu à {len(fi_derivs_list)} fonctions")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 : Gram finale sur H  (G_H_{ij} = <f_i, f_j>_Sobolev sur idx_H)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 2 : Gram finale sur H ({len(fi_derivs_list)} fonctions, n_G_H={n_H})...")

n_lev = len(fi_derivs_list)
G_H = np.zeros((n_lev, n_lev))
w0 = params["weights"].get(0,0.); w1 = params["weights"].get(1,0.)
w2 = params["weights"].get(2,0.); w3 = params["weights"].get(3,0.)

if w0:
    PHI_H = np.stack([
        fi['phi'] if fi['phi'].shape[0] == n_H else fi['phi'][idx_H]
        for fi in fi_derivs_list], axis=1)
    G_H += w0*(PHI_H.T@PHI_H)/n_H

if w1:
    idx_with_grad = [i for i, fi in enumerate(fi_derivs_list) if fi['grad'] is not None]
    if idx_with_grad:
        GRAD_H = np.stack([
            (fi_derivs_list[i]['grad'] if fi_derivs_list[i]['grad'].shape[0] == n_H
             else fi_derivs_list[i]['grad'][idx_H])
            for i in idx_with_grad], axis=1)
        G_part = np.einsum('xik,xjk->ij', GRAD_H, GRAD_H, optimize='optimal') * w1 / n_H
        G_H[np.ix_(idx_with_grad, idx_with_grad)] += G_part

if w2:
    idx_with_hess = [i for i, fi in enumerate(fi_derivs_list) if fi['hess'] is not None]
    if idx_with_hess:
        HESS_H = np.stack([
            (fi_derivs_list[i]['hess'] if fi_derivs_list[i]['hess'].shape[0] == n_H
             else fi_derivs_list[i]['hess'][idx_H])
            for i in idx_with_hess], axis=1)
        G_part = np.einsum('xikl,xjkl->ij', HESS_H, HESS_H, optimize='optimal') * w2 / n_H
        G_H[np.ix_(idx_with_hess, idx_with_hess)] += G_part

if w3:
    idx_with_third = [i for i, fi in enumerate(fi_derivs_list) if fi['third'] is not None]
    if idx_with_third:
        thirds = []
        for i in idx_with_third:
            t = fi_derivs_list[i]['third']
            thirds.append(t if t.shape[0] == n_H else t[idx_H])
        THIRD_H = np.stack(thirds, axis=1)
        G_partial = np.einsum('xiklm,xjklm->ij', THIRD_H, THIRD_H, optimize='optimal') * w3 / n_H
        for ii, i in enumerate(idx_with_third):
            for jj, j in enumerate(idx_with_third):
                G_H[i, j] += G_partial[ii, jj]

print(f"  G_H calculée ({n_H} pts)")

# ── SÉLECTION DES EXPERTS ─────────────────────────────────────────────────────
# En mode QP (semi-interpolation) il n'y a PAS de lambda : la norme H est
# l'objectif, pas une pénalité, et les contraintes de marge remplacent le terme
# de données. Le critère MSE-régularisé ci-dessous (hérité du mode mse) n'a donc
# aucun sens en QP -> on garde TOUS les experts et on laisse le QP de Phase 2
# leur attribuer leurs poids (un expert inutile reçoit un poids ~0, puisque le QP
# minimise la norme H sous contraintes).
n_loc  = len(y_train)
# QP : pas de lambda, donc pas de critère MSE-régularisé pour trier les experts.
# On les garde tous ; le QP de Phase 2 leur attribue leurs poids (un expert
# inutile reçoit ~0, puisque le QP minimise ‖u‖_H sous contraintes).
sel = np.ones(n_lev, bool); sel_idx = np.arange(n_lev)
print(f"\n  Sélection experts : aucune (QP) -> {n_lev}/{n_lev} gardés, le QP pondère.")

F_train_sel = F_train[:, sel]
F_test_sel  = F_test[:,  sel]
G_H_sel     = G_H[np.ix_(sel_idx, sel_idx)]

n_tr = F_train_sel.shape[0]
alpha_sel, feas_H, rank_H = solve_qp(F_train_sel, yH_train, G_H_sel,
                                     params["qp_margin"], verbose=True)
mask_H = np.ones(rank_H, bool)
print(f"  QP Phase 2 : faisable={feas_H}  ({rank_H} contraintes actives)")

alpha = np.zeros(n_lev); alpha[sel_idx] = alpha_sel
pred_H_train = F_train_sel @ alpha_sel
pred_H_test  = F_test_sel  @ alpha_sel
_, _, accH_tr, accH_te = method_metrics(pred_H_train, pred_H_test, None,
                                        np.zeros((1,1)), yH_train, yH_test)
mse_H = mean_squared_error(yH_test, pred_H_test)          # gardé pour info
mse_H_train = mean_squared_error(yH_train, pred_H_train)

print(f"  rang H = {mask_H.sum()}/{sel.sum()} (sur {n_lev} experts)")
print(f"  accuracy train = {accH_tr:.3f}  accuracy test = {accH_te:.3f}"
      f"   (‖u‖_H sur H)")
print(f"  alpha = {alpha.round(3)}")

# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV

print("\nCalibration RBF (baseline de comparaison uniquement, hors H)...")
gamma_grid = np.logspace(-3,2,20)
param_grid = {'alpha':[1e-8,1e-5,1e-3],'gamma':gamma_grid,'kernel':['rbf']}
kr_cv = GridSearchCV(KernelRidge(), param_grid=param_grid, cv=5,
                     scoring='neg_mean_squared_error', n_jobs=-1)
kr_cv.fit(X_train, y_train_pm1)
best_gamma = kr_cv.best_params_['gamma']
best_err_rbf = mean_squared_error(y_test_pm1, kr_cv.predict(X_test))
results = kr_cv.cv_results_; order = np.argsort(results['rank_test_score'])
print("\nTop 3 RBF :")
for i in range(3):
    idx = order[i]; g = results['param_gamma'][idx]; a = results['param_alpha'][idx]
    kr = KernelRidge(kernel='rbf', gamma=float(g), alpha=float(a))
    kr.fit(X_train, y_train_pm1)
    print(f"  #{i+1} γ={g:.4f} α={a:.0e} TEST={mean_squared_error(y_test_pm1, kr.predict(X_test)):.6f}")

poly = PolynomialFeatures(degree=min(params["deg_P"],8), include_bias=False)
ridge_poly = Ridge(alpha=1e-8); ridge_poly.fit(poly.fit_transform(X_train), y_train_pm1)
err_poly = mean_squared_error(y_test_pm1, ridge_poly.predict(poly.transform(X_test)))

# ══════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ
# ══════════════════════════════════════════════════════════════════════════════
print("\n"+"="*70); print("RÉSUMÉ"); print("="*70)
print(f"Polynomial Ridge  : {err_poly:.6f}")
print(f"RBF (γ={best_gamma:.4f})  : {best_err_rbf:.6f}")
print(f"\nAccuracy test individuelle des experts :")
for i in range(n_lev):
    acc_te_i = np.mean(np.sign(F_test[:,i]) == np.sign(yH_test))
    tag = "" if sel[i] else "  (écarté)"
    print(f"  {expert_names[i]:7s} : acc_te={acc_te_i:.3f}{tag}")
print(f"\nSobolev sous-espace H : accuracy test = {accH_te:.3f}  (rang={mask_H.sum()})")
print(f"\nweights={params['weights']}   (w0={params['weights'].get(0,0)} : ancre la constante)")
print(f"deg_P={params['deg_P']}  ε={params['epsilon']}  domaine={params['domain']}  d={params['n_ambiant']}")
print(f"QP : marge={params['qp_margin']}  bande=[{params['thres1']:.0e},{params['thres2']:.0e}]  "
      f"const_pen={params['const_pen']:.0e}")
print(f"experts : {params['n_levels']} cyl + {params['n_levels_wnd']} wnd + "
      f"poly={'OFF' if not USE_POLY_EXPERT else params['deg_poly_expert']}  "
      f"({n_lev} au total)")
if times_p1:
    print(f"temps phase 1 : {sum(times_p1):.1f}s total")

from sklearn.metrics import roc_auc_score, accuracy_score
# Décision par sign(f) : seuil 0. Labels de test en {0,1} pour les métriques.
y_test_class = (y_test > 0).astype(int)
pred_H     = F_test  @ alpha
pred_rbf   = kr_cv.predict(X_test)
pred_ridge = ridge_poly.predict(poly.transform(X_test))
print("\n" + "="*70)
print("CLASSIFICATION (décision sign(f), seuil = 0)")
print("="*70)
for nom, pred in [("Sobolev H", pred_H), ("RBF", pred_rbf), ("Ridge poly", pred_ridge)]:
    pred_class = (pred > 0).astype(int)
    acc = accuracy_score(y_test_class, pred_class)
    try:    auc = roc_auc_score(y_test_class, pred)
    except Exception: auc = float('nan')
    print(f"  {nom:12s} : accuracy={acc:.4f}  AUC={auc:.4f}")


  P_sep renormalisé sur le domaine : std=1 (facteur 2.9e+03, centre 1.77e+03)
Génération du cloud sur {Q=0} = {P_sep=0} ∪ {P_sep=ε} (deg_P=4, projection Gauss-Newton, 50/50)...
  X_train:(10, 4)  X_test:(5000, 4)  X_unlabeled:(3990, 4)
  train : nappe0=5  nappeε=5  max|résidu projection|=3.33e-16
  test  : nappe0=2500  nappeε=2500  max|résidu projection|=1.22e-15
  y_train: 5 × (−0.1)  |  5 × (+0.1)   (labels ±a_label, décision sign(f))

Phase 1 : calcul de 0 experts (0 cyl + 0 wnd C2)...

Calibration expert polynomial...
      QP: succès=True marge_atteinte=0.100 ||u||_H=0.0001
      [QP poly] base H-ortho rang 3753/3875, max|α|=8.56e-05, offset=0.001, faisable=True
  Poly15  | feat=3875 rang=  10 | loss=0.099782 (data=0.099654 reg=0.000128) | acc_tr=1.00 acc_te=1.000
  H étendu à 1 fonctions

Phase 2 : Gram finale sur H (1 fonctions, n_G_H=4000)...
  G_H calculée (4000 pts)

  Sélection experts : aucune (QP) -> 1/1 gardés, le QP pondère.
      QP: succès=True marge_atteinte=0.100 ||u

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.61847e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
